In [1]:
#Cell 1 — import
from src.protogen_preprocessing import (
    load_protogen_data,
    filter_annotation_ra_bl,
    select_relevant_measurement_columns,
    add_marker_name,
    transpose_measurement,
    merge_with_annotation,
    split_metadata_features,
    inspect_missing_values,
    inspect_low_variance_features,
    inspect_high_correlation_features,
    remove_high_correlation_features,
)
import pandas as pd
import numpy as np

In [2]:
#Cell 2 — load
data = load_protogen_data()

df_measurement = data["df_measurement"]
df_annotation = data["df_annotation"]

Protogen sheets loaded.
Measurement shape: (163, 597)
Annotation shape: (593, 63)


In [3]:
#Cell 3 — inspect annotation quickly
print(df_annotation.columns.tolist())
df_annotation.head()

['SampleId', 'Timepoint', 'Patient_ID', 'Digest', 'Study', 'IM.STEROIDS.3MONTHS', 'ACPA.POSITIVE', 'RHUEMATOID.FACTOR', 'DAS28.0M', 'DAS28.3M', 'DAS28.6M', 'DAS28.9M', 'DAS28.12M', 'DAS28.18M', 'HAQ.0M', 'HAQ.6M', 'SDAI.0M', 'SDAI.6M', 'SDAI.12M', 'BASOPHILS.0M', 'EOSINOPHILS.0M', 'HB.0M', 'LYMPHOCYTES.0M', 'MONOCYTES.0M', 'NEUTROPHILS.0M', 'PLT.0M', 'WBC.0M', 'CRP.0M', 'ESR.0M', 'FATIQUE.0M', 'PAIN.0M', 'TOTAL.SWOLLEN.0M', 'TOTAL.TENDER.0M', 'BASOPHILS.6M', 'EOSINOPHILS.6M', 'HB.6M', 'LYMPHOCYTES.6M', 'MONOCYTES.6M', 'NEUTROPHILS.6M', 'PLT.6M', 'WBC.6M', 'CRP.6M', 'FATIQUE.6M', 'PAIN.6M', 'TOTAL.SWOLLEN.6M', 'TOTAL.TENDER.6M', 'CRP.9M', 'TOTAL.SWOLLEN.9M', 'TOTAL.TENDER.9M', 'ORAL.STEROIDS.3M', 'AGE', 'RACE', 'GENDER', 'HEIGHT', 'WEIGHT', 'ALCOHOL_Y_N', 'CURENT SMOKER', 'Symp_Duration', 'Initial.Score', 'Final.Score', 'Erosive', 'Hep B serology wk 9 (IU/mL)', 'vaccine centre']


,SampleId,Timepoint,Patient_ID,Digest,Study,IM.STEROIDS.3MONTHS,ACPA.POSITIVE,RHUEMATOID.FACTOR,DAS28.0M,DAS28.3M,...,HEIGHT,WEIGHT,ALCOHOL_Y_N,CURENT SMOKER,Symp_Duration,Initial.Score,Final.Score,Erosive,Hep B serology wk 9 (IU/mL),vaccine centre
0,TAC1241_BL,BL,TAC1241,608CAC12FA322B4798E2E772CE6C9F0087B4933ADB8C3A...,TACERA,yes,No ...,Yes ...,6.91,7.02,...,NaN,56.6,No ...,Yes,137,9,18,0,NaN,NaN
1,TAC1241_M6,M6,TAC1241,608CAC12FA322B4798E2E772CE6C9F0087B4933ADB8C3A...,TACERA,yes,No ...,Yes ...,6.91,7.02,...,NaN,56.6,No ...,Yes,137,9,18,0,NaN,NaN
2,TAC1147_BL,BL,TAC1147,E856EB9AC7C85B70C9F42C4F6786775FE65DCDFDE2FB71...,TACERA,ND,ND,Yes ...,7.45,3.09,...,175,121,Yes ...,No,84,36,39,1,NaN,NaN
3,TAC1147_M6,M6,TAC1147,E856EB9AC7C85B70C9F42C4F6786775FE65DCDFDE2FB71...,TACERA,ND,ND,Yes ...,7.45,3.09,...,175,121,Yes ...,No,84,36,39,1,NaN,NaN
4,TAC1094_BL,BL,TAC1094,10433449E77D5E82E36A52EC5518583272BB0FD55D9BC4...,TACERA,ND,Yes ...,Yes ...,5.1,ND,...,190,70.1,Yes ...,Yes,155,0,NaN,0,NaN,NaN


In [4]:
#Cell 4 — filter to TACERA + BL
df_annotation_ra_bl = filter_annotation_ra_bl(df_annotation)
df_annotation_ra_bl.head()

Filtered annotation to TACERA + BL.
Filtered annotation shape: (265, 5)
Unique SampleIds: 265
Unique Patient_ID: 265
Unique Digest: 265


,SampleId,Timepoint,Patient_ID,Digest,Study
0,TAC1241_BL,BL,TAC1241,608CAC12FA322B4798E2E772CE6C9F0087B4933ADB8C3A...,TACERA
2,TAC1147_BL,BL,TAC1147,E856EB9AC7C85B70C9F42C4F6786775FE65DCDFDE2FB71...,TACERA
4,TAC1094_BL,BL,TAC1094,10433449E77D5E82E36A52EC5518583272BB0FD55D9BC4...,TACERA
5,TAC1272_BL,BL,TAC1272,620435B1ACF16E2D95C403139381CF3EBD6E5FCEE48141...,TACERA
9,TAC1096_BL,BL,TAC1096,37B41F5EC7B684D903683CBD9B437799296D96A9FE5FC3...,TACERA


In [5]:
#Cell 5 — sanity check
print(df_annotation_ra_bl["Study"].value_counts(dropna=False))
print(df_annotation_ra_bl["Timepoint"].value_counts(dropna=False))

Study
TACERA    265
Name: count, dtype: int64
Timepoint
BL    265
Name: count, dtype: int64


In [6]:
#Cell 6 - selecting relevant sample columns
#Takes fixed metadata columns from the measurement sheet. It keeps: ProteinID, GeneID, Gene Symbol, Gene Name
#Takes the relevant SampleIds from the filtered annotation table and Keeps only those matching sample columns in the measurement sheet
df_measurement_ra_bl = select_relevant_measurement_columns(df_measurement, df_annotation_ra_bl)
df_measurement_ra_bl.head()
print(df_measurement_ra_bl.shape)

Selected relevant measurement columns.
Measurement shape before: (163, 597)
Measurement shape after: (163, 269)
Number of selected sample columns: 265
(163, 269)


In [7]:
#Cell 7 - Create a unique marker name, MarkerName = Gene Symbol + "_" + ProteinID
#Right now your rows are markers, and later after transpose these marker names become the column names.
#We need it so that after the transpose, every future feature column has a clean, unique name
df_measurement_ra_bl = add_marker_name(df_measurement_ra_bl)
df_measurement_ra_bl[["ProteinID", "Gene Symbol", "MarkerName"]].head()

MarkerName column added.
Unique MarkerNames: 163 / 163


,ProteinID,Gene Symbol,MarkerName
0,104740305,XRCC6,XRCC6_104740305
1,104741891,SNRPD1,SNRPD1_104741891
2,104742995,ACTB,ACTB_104742995
3,104743232,PTBP1,PTBP1_104743232
4,104743420,FEN1,FEN1_104743420


In [8]:
#Cell 8 - Transpose the measurement dataset so that rows become columns and columns become rows. This way, every row is a sample, and every column is a marker/feature. This is the format we need for machine learning.
df_measurement_t = transpose_measurement(df_measurement_ra_bl)
df_measurement_t.head()
print(df_measurement_t.shape)

Measurement dataframe transposed.
Shape before transpose: (163, 270)
Shape after transpose: (265, 164)
(265, 164)


In [9]:
#Cell 9 - Mering the transposed measurement table with the filtered annotation table using SampleId.
df_gene_merged = merge_with_annotation(df_measurement_t, df_annotation_ra_bl)
df_gene_merged.head()
print(df_gene_merged.shape)

Merged transposed measurement with annotation.
Transposed measurement shape: (265, 164)
Filtered annotation shape: (265, 5)
Merged shape: (265, 168)
Unique SampleIds after merge: 265
Unique Patient_ID after merge: 265
Unique Digest after merge: 265
(265, 168)


In [10]:
#Cell 10 - Split metadata columns and feature columns since we want to do the cleaning on the feature columns only.
df_meta, df_features = split_metadata_features(df_gene_merged)

df_meta.head()
print(df_meta.shape)
print(df_features.shape)

Split merged dataframe into metadata and features.
Metadata shape: (265, 5)
Feature shape: (265, 163)
(265, 5)
(265, 163)


In [11]:
#Cell 10 - Sanity check on df_meta
print("Metadata shape:", df_meta.shape)
display(df_meta.head())

print("\nUnique SampleIds:", df_meta["SampleId"].nunique())
print("Unique Patient_ID:", df_meta["Patient_ID"].nunique())
print("Unique Digest:", df_meta["Digest"].nunique())

print("\nStudy values:")
print(df_meta["Study"].value_counts(dropna=False))

print("\nTimepoint values:")
print(df_meta["Timepoint"].value_counts(dropna=False))

Metadata shape: (265, 5)


,SampleId,Timepoint,Patient_ID,Digest,Study
0,TAC1241_BL,BL,TAC1241,608CAC12FA322B4798E2E772CE6C9F0087B4933ADB8C3A...,TACERA
1,TAC1147_BL,BL,TAC1147,E856EB9AC7C85B70C9F42C4F6786775FE65DCDFDE2FB71...,TACERA
2,TAC1094_BL,BL,TAC1094,10433449E77D5E82E36A52EC5518583272BB0FD55D9BC4...,TACERA
3,TAC1272_BL,BL,TAC1272,620435B1ACF16E2D95C403139381CF3EBD6E5FCEE48141...,TACERA
4,TAC1096_BL,BL,TAC1096,37B41F5EC7B684D903683CBD9B437799296D96A9FE5FC3...,TACERA



Unique SampleIds: 265
Unique Patient_ID: 265
Unique Digest: 265

Study values:
Study
TACERA    265
Name: count, dtype: int64

Timepoint values:
Timepoint
BL    265
Name: count, dtype: int64


In [12]:
#Cell 11 - Sanity check on df_features
print("Feature shape:", df_features.shape)
display(df_features.head())

print("\nDtypes summary:")
print(df_features.dtypes.value_counts())

print("\nBasic stats:")
display(df_features.describe().T.head(10))

Feature shape: (265, 163)


,XRCC6_104740305,SNRPD1_104741891,ACTB_104742995,PTBP1_104743232,FEN1_104743420,EIF4H_104743520,TNC_104745245,IGF1_104745249,FN1_104745436,IGFBP2_104745443,...,APOH_APOH_1113180421,HN1L_HN1L_1043144040,HNRNPA1_HNRNPA1_1066859223,HNRNPA2B1_HNRNPA2B1_1066866713,KDM6B_KDM6B_0172047444,KDM6B_KDM6B_1113180399,MVP_MVP_1066863544,NONO_NONO_0105507292,TMPO_TMPO_1066866329,ZNF574_ZNF574_1066529052
0,116.5,414.0,335.0,254.0,95.0,597.0,289.5,450.0,250.5,712.0,...,168.0,46.0,264.0,171.0,103.0,81.0,2064.0,172.0,134.5,103.0
1,181.5,76.0,236.5,102.5,53.0,130.0,136.0,91.0,167.0,332.0,...,243.0,37.0,103.0,111.0,44.0,61.5,9449.0,276.0,38.0,87.0
2,269.5,424.0,252.5,309.5,166.5,701.5,284.0,537.0,280.0,854.0,...,261.0,97.0,362.0,257.0,117.0,147.0,2128.0,192.0,171.0,140.0
3,159.0,194.0,799.0,156.0,56.0,345.5,271.0,242.0,419.0,654.5,...,106.0,55.5,1395.0,198.5,96.5,56.0,6695.5,473.0,88.0,137.0
4,209.0,82.0,245.0,95.0,43.0,153.0,80.0,119.0,138.0,174.5,...,63.0,83.0,79.5,66.0,292.0,541.0,4639.0,242.0,41.0,250.5



Dtypes summary:
float64    163
Name: count, dtype: int64

Basic stats:


,count,mean,std,min,25%,50%,75%,max
XRCC6_104740305,265.0,205.122642,431.213834,19.0,64.0,105.5,194.5,5157.0
SNRPD1_104741891,265.0,249.043396,294.873411,33.0,102.0,161.5,287.0,2984.0
ACTB_104742995,265.0,739.677358,1968.812775,54.0,170.0,266.0,466.0,16363.0
PTBP1_104743232,265.0,399.622642,1476.568159,37.0,99.0,159.5,287.0,22011.0
FEN1_104743420,265.0,128.215094,207.303524,14.0,47.0,71.0,124.0,2122.5
EIF4H_104743520,265.0,610.143396,1487.778251,48.5,187.0,311.0,554.0,20987.5
TNC_104745245,265.0,280.145283,328.256486,47.0,134.5,186.0,320.0,3471.0
IGF1_104745249,265.0,281.009434,263.080539,33.0,128.0,192.0,335.5,2147.0
FN1_104745436,265.0,285.462264,642.982623,93.0,151.0,181.5,255.0,7494.0
IGFBP2_104745443,265.0,761.426415,1150.431905,72.0,247.0,414.5,786.0,9509.0


In [13]:
# Cell 12 - inspect missing values
missing_summary = inspect_missing_values(df_features)

display(missing_summary.head(10))
print("Total missing values in df_features:", df_features.isna().sum().sum())

Missing values inspected.
Feature shape: (265, 163)
Total missing values: 0
Features with any missing values: 0


,feature,missing_count,missing_pct
0,XRCC6_104740305,0,0.0
1,SNRPD1_104741891,0,0.0
2,ACTB_104742995,0,0.0
3,PTBP1_104743232,0,0.0
4,FEN1_104743420,0,0.0
5,EIF4H_104743520,0,0.0
6,TNC_104745245,0,0.0
7,IGF1_104745249,0,0.0
8,FN1_104745436,0,0.0
9,IGFBP2_104745443,0,0.0


Total missing values in df_features: 0


In [14]:
# Cell 13 - inspect low-variance features
variance_summary = inspect_low_variance_features(df_features, threshold=1e-8)

display(variance_summary.head(10))

Low-variance inspection completed.
Feature shape: (265, 163)
Features with variance <= 1e-08: 0


,feature,variance
91,LYZ_1066859023,5886.078423
98,CTSG_1066859715,8560.751680
70,HIST1H4A_1066561916,8939.216752
77,CPSF6_1066836537,8993.324778
68,CXCL5_1066559611,10473.692203
32,MBP_1043138774,11851.238115
62,BCAP31_1066528482,13963.156568
87,FGA_1066858266,14404.620733
73,ZNF217_1066564363,17998.227902
158,KDM6B_KDM6B_1113180399,19614.552837


In [15]:
# Cell 14 - inspect high-correlation features
corr_matrix, high_corr_pairs = inspect_high_correlation_features(df_features, threshold=0.9)

display(high_corr_pairs.head(10))
print("Number of highly correlated pairs:", len(high_corr_pairs))

affected_features = set(high_corr_pairs["feature_1"]).union(set(high_corr_pairs["feature_2"]))
print("Unique affected features:", len(affected_features))
print(sorted(list(affected_features))[:30])

High-correlation inspection completed.
Feature shape: (265, 163)
Highly correlated pairs (>0.9): 53


,feature_1,feature_2,correlation
1370,FN1_104745436,PRTN3_1066558363,0.994970
2230,RPLP2_105481284,RPLP1_1113172457,0.989692
11028,IFNW1_1066559422,BMP7_1086924247,0.989158
23295,CXCL5_1066559611c,HIST1H1B_1066859506c,0.982312
1320,FN1_104745436,CLU_105483601,0.981188
2674,CLU_105483601,PRTN3_1066558363,0.980872
17550,BMP7_1086924247,PLVAP_1086926269,0.980452
2658,CLU_105483601,TUBB_1047887771,0.978067
2690,CLU_105483601,GNPTG_1066840773,0.976854
1354,FN1_104745436,TUBB_1047887771,0.972648


Number of highly correlated pairs: 53
Unique affected features: 30
['ACTB_104742995', 'BCAP31_1066528482', 'BMP7_1086924247', 'CALR_1066859121', 'CENPB_1066861712', 'CLU_105483601', 'CTSG_1066859715', 'CXCL5_1066559611c', 'EHD1_1043170594', 'FGA_1066858266c', 'FN1_104745436', 'GNPTG_1066840773', 'HIST1H1B_1066859506c', 'HIST1H4A_1066561916c', 'HIST2H2AA3_1066858757', 'HIST2H2BE_1066860575', 'HSPD1_1047911620c', 'IFNW1_1066559422', 'IGFBP6_1066863255', 'LMNB1_1047887413', 'LTF_1066838756', 'LYZ_1066859023', 'MBP_1043138774', 'PLVAP_1086926269', 'PRTN3_1066558363', 'RPLP1_1113172457', 'RPLP2_105481284', 'SNRPN_105510131', 'TNC_104745245', 'TUBB_1047887771']


In [16]:
# Cell 15 - remove high correlation
df_features_hc, dropped_corr_features = remove_high_correlation_features(df_features, threshold=0.9)

print("Shape after high-correlation removal:", df_features_hc.shape)
print("Removed features:", len(dropped_corr_features))
print("First removed features:", dropped_corr_features[:22])

High-correlation removal completed.
Original feature shape: (265, 163)
Reduced feature shape: (265, 141)
Removed highly correlated features: 22
Shape after high-correlation removal: (265, 141)
Removed features: 22
First removed features: ['CLU_105483601', 'SNRPN_105510131', 'LMNB1_1047887413', 'TUBB_1047887771', 'BCAP31_1066528482', 'PRTN3_1066558363', 'LTF_1066838756', 'GNPTG_1066840773', 'HIST2H2AA3_1066858757', 'LYZ_1066859023']
